In [12]:
import pandas as pd
path = r"C:/Users/yhsol/Downloads/AMZN_alpha_1min_2024-05_to_2025-05 (1).csv"
df = pd.read_csv(path, parse_dates=['Timestamp'])
df.set_index('Timestamp', inplace=True)
df = df.sort_index().between_time('09:30','16:00')
df_2min = df.resample('2T').agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last',
    'Volume': 'sum'
}).dropna()  
df=df_2min.copy()

C:\Users\yhsol\AppData\Local\Temp\ipykernel_21488\2196603593.py:6: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_2min = df.resample('2T').agg({


In [17]:
import pandas as pd
import pandas_ta as ta
import numpy as np

# Calculate median price
df['median_price'] = (df['High'] + df['Low']) / 2

# Calculate AO (Awesome Oscillator)
df['AO'] = ta.sma(df['median_price'], length=5) - ta.sma(df['median_price'], length=34)

# Drop NaNs introduced by indicator calculation
df.dropna(inplace=True)

buy_idx, sell_idx, each_return = [], [], []
holding = False
entry_price = 0
entry_index = None
equity_curve = [1]  # Start equity at 1

for i in range(2, len(df)):
    prev_ao = df['AO'].iloc[i-2]
    current_ao = df['AO'].iloc[i-1]
    current_open = df['Open'].iloc[i]

    # ENTRY: AO crosses above zero line between i-2 and i-1
    if not holding and prev_ao <= 0 and current_ao > 0:
        entry_price = current_open
        entry_index = i
        holding = True

    # EXIT: AO crosses below zero line between i-2 and i-1
    elif holding and prev_ao >= 0 and current_ao < 0:
        trade_return = (current_open - entry_price) / entry_price
        each_return.append(trade_return)
        buy_idx.append(entry_index)
        sell_idx.append(i)
        equity_curve.append(equity_curve[-1] * (1 + trade_return))
        holding = False

# Close open position at last bar
if holding:
    last_open = df['Open'].iloc[-1]
    trade_return = (last_open - entry_price) / entry_price
    each_return.append(trade_return)
    buy_idx.append(entry_index)
    sell_idx.append(len(df) - 1)
    equity_curve.append(equity_curve[-1] * (1 + trade_return))

# Calculate drawdown
equity_series = pd.Series(equity_curve)
rolling_max = equity_series.cummax()
drawdown = (equity_series - rolling_max) / rolling_max
max_drawdown = drawdown.min()

# Metrics
total_trades = len(each_return)
sum_return = sum(each_return)
cumulative_return = equity_curve[-1] - 1 if each_return else 0
avg_return_per_trade = np.mean(each_return) if each_return else None
win_rate = np.mean([r > 0 for r in each_return]) if each_return else None

# Calculate holding days (trading days only)
holding_days = [sell - buy for buy, sell in zip(buy_idx, sell_idx)]
total_holding_days = sum(holding_days)
avg_return_per_day = sum_return / total_holding_days if total_holding_days > 0 else 0

# Print results
print(f"Number of Trades: {total_trades}")
print(f"Average Return per Day: {avg_return_per_day:.2e}")
print(f"Cumulative Return: {cumulative_return:.2%}")
print(f"Max Drawdown: {max_drawdown:.2%}")
print(f"Win Rate: {win_rate:.2%}" if win_rate is not None else "N/A")

# Optional: Trades DataFrame with dates and prices
trades = pd.DataFrame({
    'Entry Time': df.index[buy_idx],
    'Exit Time': df.index[sell_idx],
    'Entry Price': df['Open'].iloc[buy_idx].values,
    'Exit Price': df['Open'].iloc[sell_idx].values,
    'Return': each_return
})

print("\nTop 5 Trades:")
print(trades.sort_values(by='Return', ascending=False).head(5))

print("\nWorst 5 Trades:")
print(trades.sort_values(by='Return').head(5))


Number of Trades: 1049
Average Return per Day: 3.49e-06
Cumulative Return: 5.11%
Max Drawdown: -21.59%
Win Rate: 35.27%

Top 5 Trades:
              Entry Time           Exit Time  Entry Price  Exit Price  \
531  2024-10-31 15:02:00 2024-11-01 11:52:00     186.0200    199.0601   
1023 2025-04-22 15:50:00 2025-04-23 10:58:00     173.1700    185.2500   
990  2025-04-09 13:20:00 2025-04-09 14:56:00     176.5735    186.8298   
851  2025-02-25 11:56:00 2025-02-26 11:08:00     207.3400    217.0400   
370  2024-09-11 11:18:00 2024-09-12 10:30:00     177.0500    184.7800   

        Return  
531   0.070101  
1023  0.069758  
990   0.058085  
851   0.046783  
370   0.043660  

Worst 5 Trades:
             Entry Time           Exit Time  Entry Price  Exit Price    Return
262 2024-08-01 15:18:00 2024-08-02 09:32:00      183.390     165.910 -0.095316
267 2024-08-02 15:52:00 2024-08-05 09:32:00      166.605     155.835 -0.064644
966 2025-04-02 15:30:00 2025-04-03 09:32:00      195.850     183.735 -

In [14]:
import pandas as pd
import numpy as np

bb_length = 20
bb_std = 2

holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
returns = []
equity_curve = [1]  # start at 1 (100%)

for i in range(len(df) - 1):
    if i < bb_length - 1:
        continue
    
    window = df['Close'].iloc[i - bb_length + 1 : i + 1]
    middle_band = window.mean()
    std_dev = window.std()
    upper_band = middle_band + bb_std * std_dev
    lower_band = middle_band - bb_std * std_dev
    close = df['Close'].iloc[i]

    if not holding_position and close < lower_band:
        entry_index = i + 1
        if entry_index >= len(df):
            break
        holding_position = True

    elif holding_position and close > upper_band:
        exit_index = i + 1
        if exit_index >= len(df):
            break

        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")

        buy_indices.append(entry_index)
        sell_indices.append(exit_index)

        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        returns.append(trade_return)

        equity_curve.append(equity_curve[-1] * (1 + trade_return))
        holding_position = False

# Close open position at last bar
if holding_position:
    buy_price = df['Open'].iloc[entry_index]
    sell_price = df['Open'].iloc[-1]
    trade_return = (sell_price - buy_price) / buy_price
    returns.append(trade_return)
    buy_indices.append(entry_index)
    sell_indices.append(len(df) - 1)
    equity_curve.append(equity_curve[-1] * (1 + trade_return))

# Calculate max drawdown
equity_series = pd.Series(equity_curve)
rolling_max = equity_series.cummax()
drawdown = (equity_series - rolling_max) / rolling_max
max_drawdown = drawdown.min()

total_trades = len(returns)
sum_return = sum(returns)
cumulative_return = equity_curve[-1] - 1 if returns else 0
avg_return_per_trade = np.mean(returns) if returns else None
win_rate = np.mean([r > 0 for r in returns]) if returns else None

# Calculate holding days (only trading days)
holding_days = [sell - buy for buy, sell in zip(buy_indices, sell_indices)]
total_holding_days = sum(holding_days)
avg_return_per_day = sum_return / total_holding_days if total_holding_days > 0 else 0

print(f"Number of Trades: {total_trades}")
print(f"Average Return per Day: {avg_return_per_day:.2e}")
print(f"Cumulative Return: {cumulative_return:.4f}")
print(f"Max Drawdown: {max_drawdown:.2%}")
print(f"Win Rate: {win_rate:.2%}" if win_rate is not None else "N/A")


Number of Trades: 500
Average Return per Day: 5.48e-06
Cumulative Return: 0.1206
Max Drawdown: -16.08%
Win Rate: 64.20%


In [13]:
import numpy as np
import pandas as pd

psar_step = 0.02
psar_max = 0.2

holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
returns = []
equity_curve = [1]  # start equity at 1

# PSAR calculation function
def calculate_parabolic_sar(df, step=0.02, max_af=0.2):
    sar = [0.0] * len(df)
    af = [step] * len(df)
    ep = [0.0] * len(df)
    trend = [True] * len(df)  # True = uptrend, False = downtrend

    trend[0] = df['Close'].iloc[0] < df['Close'].iloc[1]
    sar[0] = df['Low'].iloc[0] if trend[0] else df['High'].iloc[0]
    ep[0] = df['High'].iloc[0] if trend[0] else df['Low'].iloc[0]

    for i in range(1, len(df)):
        prev_sar = sar[i-1]
        prev_af = af[i-1]
        prev_ep = ep[i-1]

        sar[i] = prev_sar + prev_af * (prev_ep - prev_sar)

        if (trend[i-1] and sar[i] > df['Low'].iloc[i]) or (not trend[i-1] and sar[i] < df['High'].iloc[i]):
            # Reversal
            trend[i] = not trend[i-1]
            sar[i] = prev_ep
            af[i] = step
            ep[i] = df['High'].iloc[i] if trend[i] else df['Low'].iloc[i]
        else:
            trend[i] = trend[i-1]
            if (trend[i] and df['High'].iloc[i] > prev_ep) or (not trend[i] and df['Low'].iloc[i] < prev_ep):
                af[i] = min(max_af, prev_af + step)
                ep[i] = df['High'].iloc[i] if trend[i] else df['Low'].iloc[i]
            else:
                af[i] = prev_af
                ep[i] = prev_ep

    return sar, trend

sar, trend = calculate_parabolic_sar(df, psar_step, psar_max)

for i in range(1, len(df) - 1):
    # ENTRY: Buy at bullish reversal (trend down to up)
    if not holding_position and trend[i-1] == False and trend[i] == True:
        entry_index = i + 1
        if entry_index >= len(df):
            break
        holding_position = True

    # EXIT: Sell at bearish reversal (trend up to down)
    elif holding_position and trend[i-1] == True and trend[i] == False:
        exit_index = i + 1
        if exit_index >= len(df):
            break

        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")

        buy_indices.append(entry_index)
        sell_indices.append(exit_index)

        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        returns.append(trade_return)

        equity_curve.append(equity_curve[-1] * (1 + trade_return))
        holding_position = False

# Close open position at last bar
if holding_position:
    buy_price = df['Open'].iloc[entry_index]
    sell_price = df['Open'].iloc[-1]
    trade_return = (sell_price - buy_price) / buy_price
    returns.append(trade_return)
    buy_indices.append(entry_index)
    sell_indices.append(len(df) - 1)
    equity_curve.append(equity_curve[-1] * (1 + trade_return))

# Calculate drawdown
equity_series = pd.Series(equity_curve)
rolling_max = equity_series.cummax()
drawdown = (equity_series - rolling_max) / rolling_max
max_drawdown = drawdown.min()

# Metrics
total_trades = len(returns)
sum_return = sum(returns)
cumulative_return = equity_curve[-1] - 1 if returns else 0
avg_return_per_trade = np.mean(returns) if returns else None
win_rate = np.mean([r > 0 for r in returns]) if returns else None

# Calculate holding days (trading days only)
holding_days = [sell - buy for buy, sell in zip(buy_indices, sell_indices)]
total_holding_days = sum(holding_days)
avg_return_per_day = sum_return / total_holding_days if total_holding_days > 0 else 0

print(f"Number of Trades: {total_trades}")
print(f"Average Return per Day: {avg_return_per_day:.2e}")
print(f"Cumulative Return: {cumulative_return:.4f}")
print(f"Max Drawdown: {max_drawdown:.2%}")
print(f"Win Rate: {win_rate:.2%}" if win_rate is not None else "N/A")


Number of Trades: 2302
Average Return per Day: 8.70e-06
Cumulative Return: 0.2018
Max Drawdown: -15.35%
Win Rate: 38.58%


In [9]:
import numpy as np
import pandas as pd

env_length = 20
env_dev = 0.02

holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
returns = []
equity_curve = [1]  # start equity at 1

for i in range(env_length - 1, len(df) - 1):
    window = df['Close'].iloc[i - env_length + 1 : i + 1]
    sma = window.mean()
    upper_band = sma * (1 + env_dev)
    close = df['Close'].iloc[i]

    # ENTRY: Buy if price closes above upper envelope and not holding
    if not holding_position and close > upper_band:
        entry_index = i + 1
        if entry_index >= len(df):
            break
        holding_position = True

    # EXIT: Sell if price closes below SMA and holding
    elif holding_position and close < sma:
        exit_index = i + 1
        if exit_index >= len(df):
            break

        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")

        buy_indices.append(entry_index)
        sell_indices.append(exit_index)

        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        returns.append(trade_return)

        equity_curve.append(equity_curve[-1] * (1 + trade_return))
        holding_position = False

# Handle last open position
if holding_position:
    buy_price = df['Open'].iloc[entry_index]
    sell_price = df['Open'].iloc[-1]
    trade_return = (sell_price - buy_price) / buy_price
    returns.append(trade_return)
    buy_indices.append(entry_index)
    sell_indices.append(len(df) - 1)
    equity_curve.append(equity_curve[-1] * (1 + trade_return))

# Calculate drawdown
equity_series = pd.Series(equity_curve)
rolling_max = equity_series.cummax()
drawdown = (equity_series - rolling_max) / rolling_max
max_drawdown = drawdown.min()

# Metrics
total_trades = len(returns)
sum_return = sum(returns)
cumulative_return = equity_curve[-1] - 1 if returns else 0
avg_return_per_trade = np.mean(returns) if returns else None
win_rate = np.mean([r > 0 for r in returns]) if returns else None

# Calculate holding days (trading days only)
holding_days = [sell - buy for buy, sell in zip(buy_indices, sell_indices)]
total_holding_days = sum(holding_days)
avg_return_per_day = sum_return / total_holding_days if total_holding_days > 0 else 0

print(f"Number of Trades: {total_trades}")
print(f"Average Return per Day: {avg_return_per_day:.2e}")
print(f"Cumulative Return: {cumulative_return:.4f}")
print(f"Max Drawdown: {max_drawdown:.2%}")
print(f"Win Rate: {win_rate:.2%}" if win_rate is not None else "N/A")


Number of Trades: 13
Average Return per Day: 1.44e-04
Cumulative Return: 0.0479
Max Drawdown: -2.75%
Win Rate: 61.54%


In [ ]:
import numpy as np

# OBV calculation
obv = [0]
for i in range(1, len(df)):
    if df['Close'].iloc[i] > df['Close'].iloc[i-1]:
        obv.append(obv[-1] + df['Volume'].iloc[i])
    elif df['Close'].iloc[i] < df['Close'].iloc[i-1]:
        obv.append(obv[-1] - df['Volume'].iloc[i])
    else:
        obv.append(obv[-1])
df['OBV'] = obv

holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
returns = []

for i in range(1, len(df) - 1):
    # ENTRY: Buy if OBV rises above previous OBV and price rises
    if not holding_position and df['OBV'].iloc[i] > df['OBV'].iloc[i-1] and df['Close'].iloc[i] > df['Close'].iloc[i-1]:
        entry_index = i + 1  # Buy at NEXT bar's open
        if entry_index >= len(df):
            break
        holding_position = True

    # EXIT: Sell if OBV falls below previous OBV and price falls
    elif holding_position and df['OBV'].iloc[i] < df['OBV'].iloc[i-1] and df['Close'].iloc[i] < df['Close'].iloc[i-1]:
        exit_index = i + 1  # Sell at NEXT bar's open
        if exit_index >= len(df):
            break

        # Debug: check for same-bar trade
        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")

        buy_indices.append(entry_index)
        sell_indices.append(exit_index)
        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        returns.append(trade_return)

        holding_position = False

# Performance statistics
total_trades = len(returns)
sum_return = sum(returns)
cumulative_return = np.prod([1 + r for r in returns]) - 1 if returns else 0
avg_return = np.mean(returns) if returns else None
win_rate = np.mean([r > 0 for r in returns]) if returns else None

print(f"Total Trades: {total_trades}")
print(f"Sum of Returns: {sum_return:.4f}")
print(f"Cumulative Return: {cumulative_return:.4f}")
print(f"Average Return per Trade: {avg_return:.4f}" if avg_return is not None else "N/A")
print(f"Win Rate: {win_rate:.2%}" if win_rate is not None else "N/A")


Total Trades: 12217
Sum of Returns: -0.0573
Cumulative Return: -0.0862
Average Return per Trade: -0.0000
Win Rate: 34.84%


In [41]:
import numpy as np

# Calculate VWAP
tp = (df['High'] + df['Low'] + df['Close']) / 3
cum_tp_vol = (tp * df['Volume']).cumsum()
cum_vol = df['Volume'].cumsum()
df['VWAP'] = cum_tp_vol / cum_vol

holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
returns = []

for i in range(1, len(df) - 1):
    # ENTRY: Buy if price crosses above VWAP and not holding
    if not holding_position and df['Close'].iloc[i-1] < df['VWAP'].iloc[i-1] and df['Close'].iloc[i] > df['VWAP'].iloc[i]:
        entry_index = i + 1  # Buy at NEXT bar's open
        if entry_index >= len(df):
            break
        holding_position = True

    # EXIT: Sell if price crosses below VWAP and holding
    elif holding_position and df['Close'].iloc[i-1] > df['VWAP'].iloc[i-1] and df['Close'].iloc[i] < df['VWAP'].iloc[i]:
        exit_index = i + 1  # Sell at NEXT bar's open
        if exit_index >= len(df):
            break

        # Debug: check for same-bar trade
        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")

        buy_indices.append(entry_index)
        sell_indices.append(exit_index)
        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        returns.append(trade_return)

        holding_position = False

# Performance statistics
total_trades = len(returns)
sum_return = sum(returns)
cumulative_return = np.prod([1 + r for r in returns]) - 1 if returns else 0
avg_return = np.mean(returns) if returns else None
win_rate = np.mean([r > 0 for r in returns]) if returns else None

print(f"Total Trades: {total_trades}")
print(f"Sum of Returns: {sum_return:.4f}")
print(f"Cumulative Return: {cumulative_return:.4f}")
print(f"Average Return per Trade: {avg_return:.4f}" if avg_return is not None else "N/A")
print(f"Win Rate: {win_rate:.2%}" if win_rate is not None else "N/A")


Total Trades: 123
Sum of Returns: -0.1042
Cumulative Return: -0.1029
Average Return per Trade: -0.0008
Win Rate: 4.88%


In [ ]:
import numpy as np

# ADX standard parameters
adx_length = 14      # Common period for ADX
adx_threshold = 25   # Trend strength threshold

holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
returns = []

# Calculate +DM, -DM, and TR
plus_dm = [0]
minus_dm = [0]
tr = [0]
for i in range(1, len(df)):
    up_move = df['High'].iloc[i] - df['High'].iloc[i-1]
    down_move = df['Low'].iloc[i-1] - df['Low'].iloc[i]
    plus_dm.append(up_move if up_move > down_move and up_move > 0 else 0)
    minus_dm.append(down_move if down_move > up_move and down_move > 0 else 0)
    tr.append(max(
        df['High'].iloc[i] - df['Low'].iloc[i],
        abs(df['High'].iloc[i] - df['Close'].iloc[i-1]),
        abs(df['Low'].iloc[i] - df['Close'].iloc[i-1])
    ))

# Smooth +DM, -DM, and TR
plus_dm_sm = [np.mean(plus_dm[:adx_length])]
minus_dm_sm = [np.mean(minus_dm[:adx_length])]
tr_sm = [np.mean(tr[:adx_length])]
for i in range(adx_length, len(df)):
    plus_dm_sm.append((plus_dm_sm[-1]*(adx_length-1) + plus_dm[i]) / adx_length)
    minus_dm_sm.append((minus_dm_sm[-1]*(adx_length-1) + minus_dm[i]) / adx_length)
    tr_sm.append((tr_sm[-1]*(adx_length-1) + tr[i]) / adx_length)

# Calculate +DI, -DI, DX, and ADX
plus_di = 100 * np.array(plus_dm_sm) / np.array(tr_sm)
minus_di = 100 * np.array(minus_dm_sm) / np.array(tr_sm)
dx = 100 * np.abs(plus_di - minus_di) / (plus_di + minus_di)
adx = [np.mean(dx[:adx_length])]
for i in range(adx_length, len(dx)):
    adx.append((adx[-1]*(adx_length-1) + dx[i]) / adx_length)

# Align lengths for trading logic
adx = [np.nan]*(2*adx_length-1) + adx
plus_di = [np.nan]*(adx_length-1) + list(plus_di)
minus_di = [np.nan]*(adx_length-1) + list(minus_di)

for i in range(2*adx_length-1, len(df)-1):
    # ENTRY: Buy if ADX > threshold and +DI crosses above -DI
    if not holding_position and adx[i-1] <= adx_threshold and adx[i] > adx_threshold and plus_di[i] > minus_di[i]:
        entry_index = i + 1  # Buy at NEXT bar's open
        if entry_index >= len(df):
            break
        holding_position = True

    # EXIT: Sell if ADX > threshold and -DI crosses above +DI
    elif holding_position and adx[i-1] <= adx_threshold and adx[i] > adx_threshold and minus_di[i] > plus_di[i]:
        exit_index = i + 1  # Sell at NEXT bar's open
        if exit_index >= len(df):
            break

        # Debug: check for same-bar trade
        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")

        buy_indices.append(entry_index)
        sell_indices.append(exit_index)
        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        returns.append(trade_return)

        holding_position = False

# Performance statistics
total_trades = len(returns)
sum_return = sum(returns)
cumulative_return = np.prod([1 + r for r in returns]) - 1 if returns else 0
avg_return = np.mean(returns) if returns else None
win_rate = np.mean([r > 0 for r in returns]) if returns else None

print(f"Total Trades: {total_trades}")
print(f"Sum of Returns: {sum_return:.4f}")
print(f"Cumulative Return: {cumulative_return:.4f}")
print(f"Average Return per Trade: {avg_return:.4f}" if avg_return is not None else "N/A")
print(f"Win Rate: {win_rate:.2%}" if win_rate is not None else "N/A")


Number of Trades: 371
Average Return per Day: 3.48e-06
Cumulative Return: 0.1318
Max Drawdown: -21.41%
Win Rate: 39.35%


In [ ]:
import numpy as np
import pandas as pd

# Ichimoku standard parameters
tenkan_period = 9
kijun_period = 26
senkou_b_period = 52
displacement = 26

holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
returns = []
equity_curve = [1]  # Start with $1 equity

# Calculate Ichimoku components
df['tenkan_sen'] = (df['High'].rolling(window=tenkan_period).max() + df['Low'].rolling(window=tenkan_period).min()) / 2
df['kijun_sen'] = (df['High'].rolling(window=kijun_period).max() + df['Low'].rolling(window=kijun_period).min()) / 2
df['senkou_span_a'] = ((df['tenkan_sen'] + df['kijun_sen']) / 2).shift(displacement)
df['senkou_span_b'] = ((df['High'].rolling(window=senkou_b_period).max() + df['Low'].rolling(window=senkou_b_period).min()) / 2).shift(displacement)
df['chikou_span'] = df['Close'].shift(-displacement)

# Backtest loop
for i in range(senkou_b_period + displacement, len(df) - 1):
    close = df['Close'].iloc[i]
    span_a = df['senkou_span_a'].iloc[i]
    span_b = df['senkou_span_b'].iloc[i]
    tenkan = df['tenkan_sen'].iloc[i]
    kijun = df['kijun_sen'].iloc[i]

    # ENTRY condition
    if not holding_position and close > span_a and close > span_b and tenkan > kijun:
        entry_index = i + 1
        if entry_index >= len(df):
            break
        holding_position = True

    # EXIT condition
    elif holding_position and ((close < span_a and close < span_b) or tenkan < kijun):
        exit_index = i + 1
        if exit_index >= len(df):
            break

        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")

        buy_indices.append(entry_index)
        sell_indices.append(exit_index)

        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        returns.append(trade_return)

        equity_curve.append(equity_curve[-1] * (1 + trade_return))
        holding_position = False

# Handle open position at end
if holding_position:
    buy_price = df['Open'].iloc[entry_index]
    sell_price = df['Open'].iloc[-1]
    trade_return = (sell_price - buy_price) / buy_price
    returns.append(trade_return)
    buy_indices.append(entry_index)
    sell_indices.append(len(df) - 1)
    equity_curve.append(equity_curve[-1] * (1 + trade_return))

# Equity curve and drawdown
equity_series = pd.Series(equity_curve)
rolling_max = equity_series.cummax()
drawdown = (equity_series - rolling_max) / rolling_max
max_drawdown = drawdown.min()

# Summary metrics
total_trades = len(returns)
sum_return = sum(returns)
cumulative_return = equity_curve[-1] - 1 if returns else 0
avg_return_per_trade = np.mean(returns) if returns else None
win_rate = np.mean([r > 0 for r in returns]) if returns else None

# ✅ Only include trading (holding) days in avg return/day
holding_days = [sell - buy for buy, sell in zip(buy_indices, sell_indices)]
total_holding_days = sum(holding_days)
avg_return_per_day = sum_return / total_holding_days if total_holding_days > 0 else 0

# Print results
print(f"Number of Trades: {total_trades}")
print(f"Average Return per Day: {avg_return_per_day:.2e}")
print(f"Cumulative Return: {cumulative_return:.4f}")
print(f"Max Drawdown: {max_drawdown:.2%}")
print(f"Win Rate: {win_rate:.2%}" if win_rate is not None else "N/A")


Number of Trades: 1666
Average Return per Day: 5.34e-06
Cumulative Return: 0.1905
Max Drawdown: -14.98%
Win Rate: 36.55%


In [44]:
import numpy as np

# StdDev standard parameters
std_length = 20      # Common period for StdDev

holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
returns = []

for i in range(std_length, len(df) - 1):
    window = df['Close'].iloc[i - std_length + 1 : i + 1]
    sma = window.mean()
    std = window.std()
    close = df['Close'].iloc[i]

    # ENTRY: Buy if StdDev increases and price is above SMA
    if not holding_position and std > window[:-1].std() and close > sma:
        entry_index = i + 1  # Buy at NEXT bar's open
        if entry_index >= len(df):
            break
        holding_position = True

    # EXIT: Sell if StdDev increases and price is below SMA
    elif holding_position and std > window[:-1].std() and close < sma:
        exit_index = i + 1  # Sell at NEXT bar's open
        if exit_index >= len(df):
            break

        # Debug: check for same-bar trade
        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")

        buy_indices.append(entry_index)
        sell_indices.append(exit_index)
        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        returns.append(trade_return)

        holding_position = False

# Performance statistics
total_trades = len(returns)
sum_return = sum(returns)
cumulative_return = np.prod([1 + r for r in returns]) - 1 if returns else 0
avg_return = np.mean(returns) if returns else None
win_rate = np.mean([r > 0 for r in returns]) if returns else None

print(f"Total Trades: {total_trades}")
print(f"Sum of Returns: {sum_return:.4f}")
print(f"Cumulative Return: {cumulative_return:.4f}")
print(f"Average Return per Trade: {avg_return:.4f}" if avg_return is not None else "N/A")
print(f"Win Rate: {win_rate:.2%}" if win_rate is not None else "N/A")


Total Trades: 1141
Sum of Returns: -0.0370
Cumulative Return: -0.0664
Average Return per Trade: -0.0000
Win Rate: 34.88%


In [6]:
import numpy as np
import pandas as pd

# --- A/D Line Calculation ---
ad = [0]
for i in range(1, len(df)):
    high = df['High'].iloc[i]
    low = df['Low'].iloc[i]
    close = df['Close'].iloc[i]
    volume = df['Volume'].iloc[i]

    if high != low:
        mfm = ((2 * close - high - low) / (high - low))  # Money Flow Multiplier
    else:
        mfm = 0
    mfv = mfm * volume  # Money Flow Volume
    ad.append(ad[-1] + mfv)

df['AD'] = ad

# --- Strategy Execution ---
holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
returns = []
equity_curve = [1]  # Starting equity

for i in range(1, len(df) - 1):
    ad_now = df['AD'].iloc[i]
    ad_prev = df['AD'].iloc[i - 1]
    close_now = df['Close'].iloc[i]
    close_prev = df['Close'].iloc[i - 1]

    # ENTRY: A/D rising and price rising
    if not holding_position and ad_now > ad_prev and close_now > close_prev:
        entry_index = i + 1
        if entry_index >= len(df):
            break
        holding_position = True

    # EXIT: A/D falling and price falling
    elif holding_position and ad_now < ad_prev and close_now < close_prev:
        exit_index = i + 1
        if exit_index >= len(df):
            break

        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")

        buy_indices.append(entry_index)
        sell_indices.append(exit_index)

        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        returns.append(trade_return)

        equity_curve.append(equity_curve[-1] * (1 + trade_return))
        holding_position = False

# Close last position if still open
if holding_position:
    buy_price = df['Open'].iloc[entry_index]
    sell_price = df['Open'].iloc[-1]
    trade_return = (sell_price - buy_price) / buy_price
    returns.append(trade_return)
    buy_indices.append(entry_index)
    sell_indices.append(len(df) - 1)
    equity_curve.append(equity_curve[-1] * (1 + trade_return))

# --- Performance Metrics ---
equity_series = pd.Series(equity_curve)
rolling_max = equity_series.cummax()
drawdown = (equity_series - rolling_max) / rolling_max
max_drawdown = drawdown.min()

total_trades = len(returns)
sum_return = sum(returns)
cumulative_return = equity_curve[-1] - 1 if returns else 0
avg_return_per_trade = np.mean(returns) if returns else None
win_rate = np.mean([r > 0 for r in returns]) if returns else None

# ✅ Only count days when a trade is open
holding_days = [sell - buy for buy, sell in zip(buy_indices, sell_indices)]
total_holding_days = sum(holding_days)
avg_return_per_day = sum_return / total_holding_days if total_holding_days > 0 else 0

# --- Print Results ---
print(f"Number of Trades: {total_trades}")
print(f"Average Return per Day: {avg_return_per_day:.2e}")
print(f"Cumulative Return: {cumulative_return:.4f}")
print(f"Max Drawdown: {max_drawdown:.2%}")
print(f"Win Rate: {win_rate:.2%}" if win_rate is not None else "N/A")


Number of Trades: 19944
Average Return per Day: -2.38e-06
Cumulative Return: -0.1412
Max Drawdown: -32.78%
Win Rate: 35.85%


In [19]:
import numpy as np
import pandas as pd

# CMO standard parameters
cmo_length = 14

holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
returns = []

# Calculate CMO
cmo = [np.nan] * len(df)
for i in range(cmo_length, len(df)):
    window = df['Close'].iloc[i-cmo_length+1:i+1]
    diffs = window.diff().dropna()
    su = diffs[diffs > 0].sum()
    sd = -diffs[diffs < 0].sum()
    if su + sd != 0:
        cmo[i] = 100 * (su - sd) / (su + sd)
    else:
        cmo[i] = 0
df['CMO'] = cmo

# Track equity curve
equity_curve = [1]

for i in range(cmo_length, len(df) - 1):
    if not holding_position and df['CMO'].iloc[i-1] < 0 and df['CMO'].iloc[i] > 0:
        entry_index = i + 1
        if entry_index >= len(df):
            break
        holding_position = True

    elif holding_position and df['CMO'].iloc[i-1] > 0 and df['CMO'].iloc[i] < 0:
        exit_index = i + 1
        if exit_index >= len(df):
            break

        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")

        buy_indices.append(entry_index)
        sell_indices.append(exit_index)
        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        returns.append(trade_return)

        equity_curve.append(equity_curve[-1] * (1 + trade_return))
        holding_position = False

# Final performance stats
total_trades = len(returns)
sum_return = sum(returns)
cumulative_return = np.prod([1 + r for r in returns]) - 1 if returns else 0
avg_return = np.mean(returns) if returns else None
win_rate = np.mean([r > 0 for r in returns]) if returns else None

# Average return per day (assuming 1 bar = 1 day)
trading_days = (df.index[-1] - df.index[cmo_length]).days if isinstance(df.index, pd.DatetimeIndex) else len(df) - cmo_length
avg_return_per_day = (cumulative_return / trading_days) if trading_days > 0 else None

# Max Drawdown
if len(equity_curve) > 1:
    equity_curve_series = pd.Series(equity_curve)
    running_max = equity_curve_series.cummax()
    drawdown = (equity_curve_series - running_max) / running_max
    max_drawdown = drawdown.min()
else:
    max_drawdown = None

# Print results
print(f"Total Trades: {total_trades}")
print(f"Sum of Returns: {sum_return:.4f}")
print(f"Cumulative Return: {cumulative_return:.4f}")
print(f"Average Return per Trade: {avg_return:.4f}" if avg_return is not None else "N/A")
print(f"Average Return per Day: {avg_return_per_day:.2e}" if avg_return_per_day is not None else "N/A")
print(f"Win Rate: {win_rate:.2%}" if win_rate is not None else "N/A")
print(f"Max Drawdown: {max_drawdown:.2%}" if max_drawdown is not None else "N/A")


Total Trades: 2989
Sum of Returns: 0.0858
Cumulative Return: 0.0550
Average Return per Trade: 0.0000
Average Return per Day: 1.51e-04
Win Rate: 33.52%
Max Drawdown: -24.19%


In [20]:
import numpy as np
import pandas as pd

ma_length = 20
envelope_pct = 0.02

holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
returns = []
equity_curve = [1]  # Starting equity

for i in range(ma_length - 1, len(df) - 1):
    window = df['Close'].iloc[i - ma_length + 1 : i + 1]
    sma = window.mean()
    upper_band = sma * (1 + envelope_pct)
    lower_band = sma * (1 - envelope_pct)
    close = df['Close'].iloc[i]

    if not holding_position and close < lower_band:
        entry_index = i + 1  # Buy at next bar open
        if entry_index >= len(df):
            break
        holding_position = True

    elif holding_position and close > upper_band:
        exit_index = i + 1  # Sell at next bar open
        if exit_index >= len(df):
            break

        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")

        buy_indices.append(entry_index)
        sell_indices.append(exit_index)
        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        returns.append(trade_return)
        equity_curve.append(equity_curve[-1] * (1 + trade_return))
        holding_position = False

# Close open position at last bar
if holding_position:
    buy_price = df['Open'].iloc[entry_index]
    sell_price = df['Open'].iloc[-1]
    trade_return = (sell_price - buy_price) / buy_price
    returns.append(trade_return)
    buy_indices.append(entry_index)
    sell_indices.append(len(df) - 1)
    equity_curve.append(equity_curve[-1] * (1 + trade_return))

# Max Drawdown calculation
equity_series = pd.Series(equity_curve)
rolling_max = equity_series.cummax()
drawdown = (equity_series - rolling_max) / rolling_max
max_drawdown = drawdown.min()

total_trades = len(returns)
sum_return = sum(returns)
cumulative_return = equity_curve[-1] - 1 if returns else 0
avg_return_per_trade = np.mean(returns) if returns else None
win_rate = np.mean([r > 0 for r in returns]) if returns else None

# Calculate holding days
holding_days = [sell - buy for buy, sell in zip(buy_indices, sell_indices)]
total_holding_days = sum(holding_days)
avg_return_per_day = sum_return / total_holding_days if total_holding_days > 0 else 0

print(f"Number of Trades: {total_trades}")
print(f"Average Return per Day: {avg_return_per_day:.2e}")
print(f"Cumulative Return: {cumulative_return:.4f}")
print(f"Max Drawdown: {max_drawdown:.2%}")
print(f"Win Rate: {win_rate:.2%}" if win_rate is not None else "N/A")


Number of Trades: 10
Average Return per Day: 7.02e-06
Cumulative Return: 0.1221
Max Drawdown: -10.20%
Win Rate: 70.00%


In [15]:
import numpy as np
import pandas as pd

sma_length = 20

holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
returns = []
equity_curve = [1]  # Starting equity

# Calculate SMA
df['SMA'] = df['Close'].rolling(window=sma_length).mean()

for i in range(sma_length, len(df) - 1):
    close = df['Close'].iloc[i]
    sma = df['SMA'].iloc[i]

    if not holding_position and close > sma and df['Close'].iloc[i-1] <= df['SMA'].iloc[i-1]:
        entry_index = i + 1  # Buy at next bar open
        if entry_index >= len(df):
            break
        holding_position = True

    elif holding_position and close < sma and df['Close'].iloc[i-1] >= df['SMA'].iloc[i-1]:
        exit_index = i + 1  # Sell at next bar open
        if exit_index >= len(df):
            break

        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")

        buy_indices.append(entry_index)
        sell_indices.append(exit_index)
        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        returns.append(trade_return)
        equity_curve.append(equity_curve[-1] * (1 + trade_return))
        holding_position = False

# Close open position at last bar
if holding_position:
    buy_price = df['Open'].iloc[entry_index]
    sell_price = df['Open'].iloc[-1]
    trade_return = (sell_price - buy_price) / buy_price
    returns.append(trade_return)
    buy_indices.append(entry_index)
    sell_indices.append(len(df) - 1)
    equity_curve.append(equity_curve[-1] * (1 + trade_return))

# Calculate max drawdown
equity_series = pd.Series(equity_curve)
rolling_max = equity_series.cummax()
drawdown = (equity_series - rolling_max) / rolling_max
max_drawdown = drawdown.min()

total_trades = len(returns)
sum_return = sum(returns)
cumulative_return = equity_curve[-1] - 1 if returns else 0
avg_return_per_trade = np.mean(returns) if returns else None
win_rate = np.mean([r > 0 for r in returns]) if returns else None

# Calculate holding days (trading days)
holding_days = [sell - buy for buy, sell in zip(buy_indices, sell_indices)]
total_holding_days = sum(holding_days)
avg_return_per_day = sum_return / total_holding_days if total_holding_days > 0 else 0

print(f"Number of Trades: {total_trades}")
print(f"Average Return per Day: {avg_return_per_day:.2e}")
print(f"Cumulative Return: {cumulative_return:.4f}")
print(f"Max Drawdown: {max_drawdown:.2%}")
print(f"Win Rate: {win_rate:.2%}" if win_rate is not None else "N/A")


Number of Trades: 2923
Average Return per Day: 6.20e-06
Cumulative Return: 0.1303
Max Drawdown: -21.02%
Win Rate: 25.38%


In [4]:
import numpy as np
import pandas as pd

# STD strategy parameters
std_length = 20

holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
returns = []
equity_curve = [1]  # Start with $1 equity

for i in range(std_length, len(df) - 1):
    window = df['Close'].iloc[i - std_length + 1 : i + 1]
    sma = window.mean()
    std = window.std()
    prev_std = window[:-1].std()
    close = df['Close'].iloc[i]

    # ENTRY: STD rising and price above SMA
    if not holding_position and std > prev_std and close > sma:
        entry_index = i + 1
        if entry_index >= len(df):
            break
        holding_position = True

    # EXIT: STD rising and price below SMA
    elif holding_position and std > prev_std and close < sma:
        exit_index = i + 1
        if exit_index >= len(df):
            break

        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")

        buy_indices.append(entry_index)
        sell_indices.append(exit_index)

        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        returns.append(trade_return)

        equity_curve.append(equity_curve[-1] * (1 + trade_return))
        holding_position = False

# Handle final open trade
if holding_position:
    buy_price = df['Open'].iloc[entry_index]
    sell_price = df['Open'].iloc[-1]
    trade_return = (sell_price - buy_price) / buy_price
    returns.append(trade_return)
    buy_indices.append(entry_index)
    sell_indices.append(len(df) - 1)
    equity_curve.append(equity_curve[-1] * (1 + trade_return))

# Equity curve and drawdown
equity_series = pd.Series(equity_curve)
rolling_max = equity_series.cummax()
drawdown = (equity_series - rolling_max) / rolling_max
max_drawdown = drawdown.min()

# Summary metrics
total_trades = len(returns)
sum_return = sum(returns)
cumulative_return = equity_curve[-1] - 1 if returns else 0
avg_return_per_trade = np.mean(returns) if returns else None
win_rate = np.mean([r > 0 for r in returns]) if returns else None

# ✅ New: calculate only trading days (when in a position)
holding_days = [sell - buy for buy, sell in zip(buy_indices, sell_indices)]
total_holding_days = sum(holding_days)
avg_return_per_day = sum_return / total_holding_days if total_holding_days > 0 else 0

# Print results
print(f"Number of Trades: {total_trades}")
print(f"Average Return per Day: {avg_return_per_day:.2e}")
print(f"Cumulative Return: {cumulative_return:.4f}")
print(f"Max Drawdown: {max_drawdown:.2%}")
print(f"Win Rate: {win_rate:.2%}" if win_rate is not None else "N/A")


Number of Trades: 2249
Average Return per Day: 6.92e-06
Cumulative Return: 0.3624
Max Drawdown: -16.49%
Win Rate: 35.22%


In [13]:
df_2min.head()

,Open,High,Low,Close
Timestamp,,,,
2024-05-01 09:30:00,181.6350,182.000,179.5907,179.650
2024-05-01 09:32:00,179.6603,179.680,178.2600,178.530
2024-05-01 09:34:00,178.5700,180.210,178.4700,180.060
2024-05-01 09:36:00,180.0550,180.400,179.3200,180.210
2024-05-01 09:38:00,180.2000,180.785,179.9000,180.115
